In [1]:
# ============================================================
# 1️⃣ Imports and constants
# ============================================================
import meep as mp
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import os, h5py, time, concurrent.futures
from datetime import datetime
import math
import multiprocessing

# Physical constants
c0 = 299792458  # m/s
um_scale = 1e-6  # 1 µm in m

# ============================================================
# 2️⃣ Interactive configuration panel (with sweep text fields)
# ============================================================

# Basic choices (kept toggle-friendly for backward compat)
tuned_geometry = widgets.Checkbox(value=False, description="Use tuned geometry")
run_sweep = widgets.Checkbox(value=False, description="Run tuner sweep")
harminv_analysis = widgets.Checkbox(value=False, description="Run Harminv")
save_results = widgets.Checkbox(value=True, description="Save results")

disk_radius = widgets.FloatText(value=3.5, description="Disk radius (µm)")
gap = widgets.FloatText(value=0.1, description="Gap (µm)")
gap_tune = widgets.FloatText(value=0.2, description="Tuner gap (µm)")
f_thz = widgets.FloatText(value=322.0, description="Center frequency (THz)")
df_thz = widgets.FloatText(value=20.0, description="Bandwidth (THz)")
resolution = widgets.IntText(value=20, description="Resolution (px/µm)")
field_decay = widgets.FloatText(value=1e-3, description="Field decay")

# New sweep UI (text input for param name & comma-separated values)
sweep_param_name = widgets.Text(value="", placeholder="e.g. sigma_meep or resolution", description="Param name")
sweep_values_text = widgets.Text(value="", placeholder="e.g. 0.05,0.1,0.2", description="Values (csv)")

# How many parallel workers (be conservative by default)
max_workers = widgets.IntSlider(value=min(4, max(1, multiprocessing.cpu_count()-1)),
                                min=1, max=max(16, multiprocessing.cpu_count()),
                                step=1, description="Max workers")

config_box = widgets.VBox([
    widgets.HTML("<h3>Simulation Configuration</h3>"),
    widgets.HBox([tuned_geometry, run_sweep, harminv_analysis, save_results]),
    disk_radius, gap, gap_tune, f_thz, df_thz, resolution, field_decay,
    widgets.HTML("<b>Parameter sweep (leave empty to skip parallel sweep)</b>"),
    sweep_param_name, sweep_values_text, max_workers
])
display(config_box)

# ============================================================
# 3️⃣ Geometry helpers (stateless versions that take params)
# ============================================================

# Default materials (these will be recreated per-worker if material loss is swept)
def gaas_medium_from_sigma(sigma_meep=None):
    # If sigma_meep is None -> lossless default
    if sigma_meep is None:
        return mp.Medium(epsilon=12)
    # Example D_conductivity mapping (as in your snippet):
    # D_conductivity = 2*pi*1*sigma_meep/3.4  (units inherited from user code)
    return mp.Medium(epsilon=12, D_conductivity=2 * math.pi * 1 * float(sigma_meep) / 3.4)

air = mp.Medium(epsilon=1)
wg_length = 20
wg_width = 0.22
tuner_width = 0.1
pml_layers = [mp.PML(2.0)]

def arc_prism_radius_width(radius, width, angle_start, angle_end, npoints=64, material=None):
    # returns a Prism with vertices (stateless)
    r_in = radius - width/2
    r_out = radius + width/2
    outer = [mp.Vector3(r_out*np.cos(a), r_out*np.sin(a)) for a in np.linspace(angle_start, angle_end, npoints)]
    inner = [mp.Vector3(r_in*np.cos(a), r_in*np.sin(a)) for a in np.linspace(angle_end, angle_start, npoints)]
    return mp.Prism(vertices=outer + inner, height=mp.inf, material=material)

def make_geometry_from_params(params):
    """Return a geometry list based on explicit params dict (pure function)."""
    disk_r = float(params.get("disk_radius", 3.5))
    gap_v = float(params.get("gap", 0.1))
    use_tuner = bool(params.get("use_tuner", False))
    gap_tune_val = float(params.get("gap_tune", 0.2))
    sigma = params.get("sigma_meep", None)
    # material for blocks/cylinder
    mat = gaas_medium_from_sigma(sigma)

    base = [
        mp.Cylinder(radius=disk_r, height=mp.inf, material=mat, center=mp.Vector3()),
        mp.Block(size=mp.Vector3(wg_length, wg_width, mp.inf),
                 center=mp.Vector3(0, disk_r + gap_v + wg_width/2),
                 material=mat),
        mp.Block(size=mp.Vector3(wg_length, wg_width, mp.inf),
                 center=mp.Vector3(0, -disk_r - gap_v - wg_width/2),
                 material=mat)
    ]
    if use_tuner:
        base.append(arc_prism_radius_width(radius=disk_r + gap_tune_val,
                                           width=tuner_width,
                                           angle_start=-np.pi/3,
                                           angle_end=np.pi/3,
                                           npoints=128,
                                           material=mat))
    return base

# ============================================================
# 4️⃣ Worker-based simulation helper (picklable)
# ============================================================

def run_sim_worker(params):
    """
    Worker function for a single simulation. 
    Includes optional Harminv analysis if params["harminv"] = True.
    """
    # Local copies from params
    disk_r = float(params.get("disk_radius", 3.5))
    gap_v = float(params.get("gap", 0.1))
    gap_tune_val = float(params.get("gap_tune", 0.2))
    use_tuner = bool(params.get("use_tuner", False))
    resolution_local = int(params.get("resolution", 20))
    field_decay_local = float(params.get("field_decay", 1e-3))
    f_thz = float(params.get("f_thz", 322.0))
    df_thz = float(params.get("df_thz", 20.0))
    sigma_meep_val = params.get("sigma_meep", None)
    run_harminv = bool(params.get("harminv", False))

    # Build materials and geometry locally
    gaas_local = gaas_medium_from_sigma(sigma_meep_val)
    geometry = make_geometry_from_params({
        "disk_radius": disk_r,
        "gap": gap_v,
        "use_tuner": use_tuner,
        "gap_tune": gap_tune_val,
        "sigma_meep": sigma_meep_val
    })

    # Convert frequencies (THz → Meep)
    f_cen = f_thz * um_scale * 1e12 / c0
    df = df_thz * um_scale * 1e12 / c0

    # Source location
    source_x = -wg_length/2 + 4
    source_y = disk_r + gap_v + wg_width/2

    sources = [mp.Source(mp.GaussianSource(frequency=f_cen, fwidth=df),
                         component=mp.Ez,
                         center=mp.Vector3(source_x, source_y),
                         size=mp.Vector3(0, wg_width, 0))]

    cell = mp.Vector3(wg_length + 1, 2*(disk_r + gap_v + wg_width/2) + 10, 0)

    flux_bus = mp.FluxRegion(center=mp.Vector3(wg_length/2 - 4, disk_r + gap_v + wg_width/2),
                             size=mp.Vector3(0, wg_width, 0))
    flux_drop = mp.FluxRegion(center=mp.Vector3(-wg_length/2 + 4, -disk_r - gap_v - wg_width/2),
                              size=mp.Vector3(0, wg_width, 0))

    # Simulation object
    sim = mp.Simulation(cell_size=cell,
                        geometry=geometry,
                        sources=sources,
                        boundary_layers=pml_layers,
                        resolution=resolution_local)

    trans_flux_bus = sim.add_flux(f_cen, df, 1000, flux_bus)
    trans_flux_drop = sim.add_flux(f_cen, df, 1000, flux_drop)

    t0 = time.time()
    sim.run(until_after_sources=mp.stop_when_fields_decayed(100, mp.Ez, mp.Vector3(), field_decay_local))
    runtime = time.time() - t0

    # Harminv analysis (optional)
    harminv_results = None
    if run_harminv:
        print("Running Harminv analysis...")
        sim.reset_meep()
        sim = mp.Simulation(cell_size=cell,
                            geometry=geometry,
                            sources=[mp.Source(mp.ContinuousSource(frequency=f_cen),
                                               component=mp.Ez,
                                               center=mp.Vector3(source_x, source_y),
                                               size=mp.Vector3(0, wg_width, 0))],
                            boundary_layers=pml_layers,
                            resolution=resolution_local)
        sim.run(until=200)  # allow steady-state excitation
        harminv_instance = mp.Harminv(mp.Ez, mp.Vector3(), f_cen, df)
        sim.run(harminv_instance, until=200)
        harminv_results = harminv_instance.modes

    # Extract flux data
    freqs = np.array(mp.get_flux_freqs(trans_flux_bus))
    flux_bus_data = np.array(mp.get_fluxes(trans_flux_bus))
    flux_drop_data = np.array(mp.get_fluxes(trans_flux_drop))

    # --- Save everything ---
    tag_parts = [f"sim", f"r{disk_r}", f"g{gap_v}", f"f{int(f_thz)}",
                 f"res{resolution_local}", f"decay{field_decay_local}"]
    if sigma_meep_val is not None:
        tag_parts.append(f"sigma{sigma_meep_val}")
    folder_name = datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + "_".join(tag_parts)
    run_dir = os.path.join("data", folder_name)
    os.makedirs(run_dir, exist_ok=True)

    np.save(os.path.join(run_dir, "freqs.npy"), freqs)
    np.save(os.path.join(run_dir, "flux_bus.npy"), flux_bus_data)
    np.save(os.path.join(run_dir, "flux_drop.npy"), flux_drop_data)

    # Save Harminv results (if any)
    if harminv_results:
        with open(os.path.join(run_dir, "harminv_results.txt"), "w") as f:
            f.write("# freq (Meep units)\tQ\tamp\terror\tdecay\tfreq (THz)\n")
            for m in harminv_results:
                f_thz_mode = m.freq * c0 / um_scale / 1e12
                f.write(f"{m.freq:.6e}\t{m.Q:.3e}\t{abs(m.amplitude):.3e}\t{m.error:.3e}\t{m.decay:.3e}\t{f_thz_mode:.3f}\n")

    # Save params
    params_saved = dict(params)
    params_saved["runtime_seconds"] = runtime
    params_saved["run_dir"] = run_dir
    with open(os.path.join(run_dir, "params.txt"), "w") as f:
        for k, v in params_saved.items():
            f.write(f"{k} = {v}\n")

    return {
        "sweep_value": params.get("sweep_value"),
        "run_dir": run_dir,
        "freqs_file": os.path.join(run_dir, "freqs.npy"),
        "flux_bus_file": os.path.join(run_dir, "flux_bus.npy"),
        "flux_drop_file": os.path.join(run_dir, "flux_drop.npy"),
        "params": params_saved
    }


# ============================================================
# 5️⃣ Master runner that does serial or parallel sweep
# ============================================================

def parse_csv_values(text):
    """Parse a comma-separated list of numbers/strings into python list"""
    if text is None:
        return []
    vals = [v.strip() for v in text.split(",") if v.strip() != ""]
    parsed = []
    for v in vals:
        # try to parse as float or int, otherwise keep string
        try:
            if "." in v or "e" in v.lower():
                parsed.append(float(v))
            else:
                parsed.append(int(v))
        except Exception:
            # fallback to string
            parsed.append(v)
    return parsed

def run_configured_sim():
    """
    Master entry: either single-run or sweep-run. If sweep is requested and
    sweep_param_name non-empty, runs parallel/sweep mode using ProcessPoolExecutor.
    """
    print("\nStarting simulation controller...")

    # base parameters captured from current widget state
    base_params = {
        "disk_radius": float(disk_radius.value),
        "gap": float(gap.value),
        "gap_tune": float(gap_tune.value),
        "use_tuner": bool(tuned_geometry.value),
        "resolution": int(resolution.value),
        "field_decay": float(field_decay.value),
        "f_thz": float(f_thz.value),
        "df_thz": float(df_thz.value),
    }

    # If user didn't request sweep, run single sim (serial)
    if not run_sweep.value or sweep_param_name.value.strip() == "":
        print("→ Single run (no sweep) selected.")
        # Use run_sim_worker but in-process to avoid pickling overhead
        summary = run_sim_worker({**base_params, "sweep_tag": "single"})
        # Load and plot
        freqs = np.load(summary["freqs_file"])
        flux_bus = np.load(summary["flux_bus_file"])
        flux_drop = np.load(summary["flux_drop_file"])
        freq_thz = freqs * c0 / um_scale / 1e12
        plt.figure(figsize=(9,5))
        plt.plot(freq_thz, flux_bus, label="Bus")
        plt.plot(freq_thz, flux_drop, label="Drop")
        plt.xlabel("Frequency (THz)")
        plt.ylabel("Flux")
        plt.title("Single run spectra")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.show()
        print("Saved run in:", summary["run_dir"])
        return

    # Else: sweep requested
    param_name = sweep_param_name.value.strip()
    value_list = parse_csv_values(sweep_values_text.value)
    if len(value_list) == 0:
        print("No sweep values provided — aborting sweep.")
        return

    print(f"→ Parallel sweep requested on parameter '{param_name}' with {len(value_list)} values.")
    print("Max workers:", max_workers.value)

    # Build list of param dicts to run
    jobs = []
    for v in value_list:
        p = dict(base_params)  # shallow copy
        # Allow sweeping strings (e.g., different material labels) or convert numeric
        p[param_name] = v
        p["sweep_value"] = v
        p["sweep_tag"] = f"{param_name}={v}"
        jobs.append(p)

    # Run in parallel using ProcessPoolExecutor
    results = []
    with concurrent.futures.ProcessPoolExecutor(max_workers=int(max_workers.value)) as executor:
        # submit all jobs
        futures = {executor.submit(run_sim_worker, job): job for job in jobs}
        for future in concurrent.futures.as_completed(futures):
            job = futures[future]
            try:
                summary = future.result()
                results.append(summary)
                print(f"[DONE] {job.get('sweep_tag')}: saved to {summary['run_dir']}")
            except Exception as e:
                print(f"[ERROR] job {job.get('sweep_tag')} failed: {e}")

    # After all jobs complete — aggregate and plot
    if len(results) == 0:
        print("No successful results to plot.")
        return

    # sort by sweep value when numeric
    def sort_key(s):
        v = s.get("sweep_value")
        try:
            return float(v)
        except Exception:
            return str(v)
    results_sorted = sorted(results, key=sort_key)

    plt.figure(figsize=(10,6))
    for res in results_sorted:
        freqs = np.load(res["freqs_file"])
        flux_drop = np.load(res["flux_drop_file"])
        freq_thz = freqs * c0 / um_scale / 1e12
        label = str(res.get("sweep_value"))
        plt.plot(freq_thz, flux_drop, label=label)
    plt.xlabel("Frequency (THz)")
    plt.ylabel("Flux (drop port)")
    plt.title(f"Sweep results for '{param_name}'")
    plt.legend(title=param_name, bbox_to_anchor=(1.05,1), loc='upper left')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    print("Sweep finished. Results saved to 'data/' folders (one per run).")

# ============================================================
# 6️⃣ Display run button
# ============================================================
run_button = widgets.Button(description="▶ Run Simulation", button_style="success")
output_box = widgets.Output()

def on_run_click(b):
    with output_box:
        output_box.clear_output(wait=True)
        run_configured_sim()

run_button.on_click(on_run_click)
display(widgets.VBox([run_button, output_box]))


Using MPI version 3.1, 1 processes
